### Make the root project directory the access point for 'os'

In [1]:
import sys
from pathlib import Path
import os
sys.path.append(str(Path(os.getcwd()).parent))
Path(os.getcwd())

PosixPath('/mnt/dell_storage/homefolders/librad.laureateinstitute.org/mferguson/Digital-Twins/notebooks')

### Find size of entire valid patient population

In [2]:
from pathlib import Path

print(len(list(Path('/media/studies/ehr_study/analysis/mferguson/sliced_patient_json/').glob("*.json"))))

9725


### Find TRD positive and negative counts

In [3]:
from pathlib import Path
import os

from dotenv import load_dotenv
load_dotenv('../.env')

from scripts.digital_twins.predictions.trd_predictor import TRDPredictor

predictor = TRDPredictor()

positive_count = 0
negative_count = 0
for p in Path(os.environ['SLICED_PATIENT_JSON_DIR']).glob("*.json"):
    pid = p.stem
    positive_count += predictor.get_trd_status(pid)
    negative_count += 1 - predictor.get_trd_status(pid)

print(f"Positive count: {positive_count}\nNegative count: {negative_count}")

Positive count: 1034
Negative count: 8691


### Investigate metrics on patients

In [4]:
import os
from pathlib import Path
import pandas as pd
import json

from scripts.digital_twins.predictions.trd_predictor import TRDPredictor

from dotenv import load_dotenv
load_dotenv()

vital_keys = [
    "SystolicBloodPressure", 
    "DiastolicBloodPressure", 
    "PulseRate", 
    "RespirationRate",
    "WeightInGrams", 
    "HeightInInches", 
    "TemperatureInFahrenheit", 
    "BodyMassIndex"
]

predictor = TRDPredictor()
patient_infos = []
for p in Path(os.environ['SLICED_PATIENT_JSON_DIR']).glob("*.json"):
    with open(p, 'r') as f:
        vitals_to_check_for_patient = set(vital_keys)
        patient_json = json.load(f)
        patient_info = {
            "patient_id": patient_json['patient_id'],
            "mdd_to_anchor_days": patient_json['mdd_to_anchor_days'],
            'num_encounters': len(patient_json['encounters']),
            'pre_anchor_history_days': patient_json['pre_anchor_history_days'],
            'trd_label': predictor.get_trd_status(patient_json['patient_id'])
        }
        
        # Now go through each encounter and see if any of the vital keys are matched - remove any that are so we do not recheck
        for encounter in patient_json['encounters']:
            for vital in encounter['vitals']:
                for vital_key in list(vitals_to_check_for_patient):
                    temp = vital.get(vital_key)
                    if (isinstance(temp, float) or isinstance(temp, int)) and temp > 0:
                        vitals_to_check_for_patient.discard(vital_key)
        
        for key in vital_keys:
            patient_info[f"missing_{key}"] = key in vitals_to_check_for_patient
        patient_infos.append(patient_info)
                
df = pd.DataFrame(patient_infos)
df.head()

,patient_id,mdd_to_anchor_days,num_encounters,pre_anchor_history_days,trd_label,missing_SystolicBloodPressure,missing_DiastolicBloodPressure,missing_PulseRate,missing_RespirationRate,missing_WeightInGrams,missing_HeightInInches,missing_TemperatureInFahrenheit,missing_BodyMassIndex
0,859583BC32EBEE0476C127679677A7D2,0,10,1330,0,True,True,True,True,True,True,True,True
1,B31487E3C6C47AEFFD079148B89DFF7C,59,18,1091,0,False,False,False,True,False,False,False,False
2,9EF10EFC520C572E223D69866E5AEAAE,631,13,848,1,False,False,False,True,False,False,False,False
3,EC996742DEFB3C68648CB3D09B6B9072,103,8,1423,0,True,True,True,True,True,True,True,True
4,E98075DF8BBEBC0E93B6EDE1185FE809,1,7,2504,0,True,True,True,True,False,False,True,False


In [5]:
missing_percentage_df = (df.filter(like='missing_').mean() * 100).sort_values(ascending=False)
missing_percentage_df # Most missing vitals first, in descending order

missing_RespirationRate            79.516710
missing_TemperatureInFahrenheit    71.753213
missing_PulseRate                  69.984576
missing_DiastolicBloodPressure     69.244216
missing_SystolicBloodPressure      69.244216
missing_HeightInInches             64.503856
missing_BodyMassIndex              63.506427
missing_WeightInGrams              63.485861
dtype: float64

In [6]:
filtered = df.filter(regex='^(trd_label|missing_)') # Grab label and the missing vitals labels (9 columns)
# For each label, see the difference in proportions of TRD positive vs TRD negative patients who are missing
missing_difference = filtered.groupby('trd_label').mean()\
    .mul(100)\
        .T\
            .rename(columns={0: 'trd_neg_pct', 1: 'trd_pos_pct'})\
                .assign(delta_pct=lambda x: x['trd_pos_pct'] - x['trd_neg_pct'])\
                    .sort_values('delta_pct', key=abs, ascending=False)
# Positive means more TRD are missing, negative means more non TRD are missing
missing_difference

trd_label,trd_neg_pct,trd_pos_pct,delta_pct
missing_WeightInGrams,62.432401,72.340426,9.908024
missing_BodyMassIndex,62.455414,72.340426,9.885012
missing_HeightInInches,63.467955,73.210832,9.742876
missing_SystolicBloodPressure,68.335059,76.885880,8.550821
missing_DiastolicBloodPressure,68.335059,76.885880,8.550821
missing_PulseRate,69.117478,77.272727,8.155249
missing_TemperatureInFahrenheit,71.073524,77.466151,6.392627
missing_RespirationRate,78.840179,85.203095,6.362915
